# HeatUp 02 - compute (optional)

Runs everything the plans ask for (`plan.json` of every phase, written by notebook 01) that is not done yet, with the potential chosen in
notebook 01: `relax`, `elastic`, `harmonic`, `quasiharmonic`, `perturbative`, `md`, in dependency order.

* **Nothing is configured here.** Safe to run again: finished work is never repeated, an interrupted MD resumes, a failed task is retried
  and never stops the others. Each task runs in its own process, so GPU memory is released.
* **Skip this notebook** if you compute with another code (DFT): notebook 03 only needs the files.

In [ ]:
import os
import warnings
import pandas as pd
from IPython.display import display

import heatup
from heatup import cfg, run_plan, status
from heatup.analyze.report import show_status, status_table
from heatup.core.ledger import manifest_summary

warnings.filterwarnings("ignore")
heatup.verbose()
if os.path.exists("heatup_project.json"):
    heatup.load_json("heatup_project.json")  # folder roots and calculator written by notebook 01
print("HeatUp", heatup.__version__, "|", cfg.phases.database_root)
print("calculator:", cfg.calc.backend, cfg.calc.model, "| dtype", cfg.calc.dtype, "| device", cfg.calc.device)
# Only on purpose (a different calculator makes finished steps stale):  cfg.calc.device = "cpu"  |  cfg.run.force = ["md"] or ["all"]

## What is left to do
`done` is finished; anything else will run (`calc=True` also compares the calculator).

In [ ]:
show_status(status(calc=True))
todo = run_plan(dry_run=True)
display(pd.DataFrame(todo).query("action != 'done'").set_index("phase"))

## Run

In [ ]:
results = run_plan()
display(pd.DataFrame(results).groupby("action").size().rename("tasks").to_frame())

## Afterwards

In [ ]:
show_status(status(calc=True))
display(status_table(status(calc=True), problems_only=True))
tag = results[0]["phase"]  # every finished step stores the exact options it used
print(manifest_summary(os.path.join(cfg.phases.database_root, tag), "relax"))

### Other protocols
`run_plan` is one loop, so a cheap first stage you inspect before paying for the rest is just several calls:
```python
run_plan(steps=["relax", "elastic"])          # only these steps (of every phase)
status()                                      # inspect, edit plan.json, drop phases ...
run_plan()                                    # ... then everything else
run_plan(targets=["database/BaTiO3/P4mm"])    # only some targets (and their competitors)
```
`cfg.run.force` (or `with override(**{"run.force": ["relax"]}): run_plan()`) recomputes steps whatever the cache says.